In [1]:
# --- Uniform fog ramp video generator (3s @ 5 FPS) ---------------------------
# Usage:
#   image_paths = ["data/scene01.jpg", "data/scene02.png"]
#   make_uniform_fog_videos(image_paths, out_dir="videos", seconds=3, fps=5, max_strength=1.0)

import os
import sys
from pathlib import Path
import numpy as np
import cv2

# Ensure we can import your module: simulation/Weather_artifacts/uniform_fog.py
# Adjust the project_root to wherever "simulation" lives in your environment.
project_root = Path.cwd()  # or Path("/path/to/your/project")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from uniform_fog import apply_uniform_fog  # your code

def make_uniform_fog_video(
    image_path: str | Path,
    out_path: str | Path,
    seconds: int = 3,
    fps: int = 5,
    max_strength: float = 1.0,
):
    """
    Create a fog-ramp video for a single image.
    - Duration: `seconds`
    - FPS: `fps`
    - Fog strength ramps linearly from 0 -> max_strength
    """
    image_path = Path(image_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Load image (BGR)
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    h, w = img.shape[:2]
    total_frames = int(seconds * fps)

    # OpenCV writer (MP4V)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError("Could not open VideoWriter. Try a different FOURCC or path.")

    # Ramp fog strength 0 -> max_strength
    strengths = np.linspace(0.0, max_strength, total_frames, dtype=np.float32)

    for s in strengths:
        fogged, _ = apply_uniform_fog(img, float(s))  # returns (fogged_bgr_uint8, k_map_float32)
        writer.write(fogged)

    writer.release()
    print(f"[OK] Wrote: {out_path} ({seconds}s @ {fps} FPS)")

def make_uniform_fog_videos(
    image_paths: list[str | Path],
    out_dir: str | Path = "videos",
    seconds: int = 3,
    fps: int = 5,
    max_strength: float = 1.0,
):
    """
    Make one fog-ramp video per input image.
    Output files: <stem>_uniformfog.mp4 in out_dir.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in image_paths:
        p = Path(p)
        out_path = out_dir / f"{p.stem}_uniformfog.mp4"
        make_uniform_fog_video(p, out_path, seconds=seconds, fps=fps, max_strength=max_strength)

# --- Example: run on one or more images -------------------------------------
image_paths = ["images/example1.png"]  # or multiple paths
make_uniform_fog_videos(image_paths, out_dir="videos", seconds=5, fps=10, max_strength=0.7)


[OK] Wrote: videos\example1_uniformfog.mp4 (5s @ 10 FPS)


In [4]:
%pip install noise

Defaulting to user installation because normal site-packages is not writeable
  Using cached noise-1.2.2.zip (132 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for noise: filename=noise-1.2.2-cp312-cp312-win_amd64.whl size=31961 sha256=226d6a94dc6d4ad70d0a1ed2f8c37baee36ad06497f325282a95bfc17276f30f
  Stored in directory: c:\users\mosel\appdata\local\pip\cache\wheels\78\71\a2\47a0c6acdeb8f7a2f4e69067d3c737219e36414136441a1ef8
Successfully built noise
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# --- examples/example_vid.ipynb ---------------------------------------------
# Stacked fog ramp video: Uniform (top) + Heterogeneous (bottom)
# Output: examples/videos/stack_uniform_vs_hetero.mp4  (3s @ 5 FPS)

from pathlib import Path
import numpy as np
import cv2

# Imports: modules are in the SAME folder as this notebook
from uniform_fog import apply_uniform_fog
from hetero_fog import apply_heterogeneous_fog

def _read_bgr(p: Path) -> np.ndarray:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {p}")
    return img

def _ramp(seconds: int, fps: int, max_strength: float) -> np.ndarray:
    return np.linspace(0.0, float(max_strength), int(seconds * fps), dtype=np.float32)

def _resize_to_width(img: np.ndarray, target_w: int) -> np.ndarray:
    h, w = img.shape[:2]
    if w == target_w: 
        return img
    new_h = int(round(h * (target_w / w)))
    return cv2.resize(img, (target_w, new_h), interpolation=cv2.INTER_AREA)

def make_stacked_uniform_vs_hetero_video(
    image_top: Path,            # image for uniform fog
    image_bottom: Path,         # image for heterogeneous fog
    out_video: Path,            # output mp4 path
    seconds: int = 3,
    fps: int = 5,
    max_uniform: float = 0.7,
    max_hetero: float = 0.7,
    also_save_individual: bool = True,
):
    # I/O
    out_video.parent.mkdir(parents=True, exist_ok=True)
    img_top = _read_bgr(image_top)
    img_bot = _read_bgr(image_bottom)

    # Strength ramps
    s_u = _ramp(seconds, fps, max_uniform)
    s_h = _ramp(seconds, fps, max_hetero)

    # Generate frames
    frames_u = [apply_uniform_fog(img_top, float(s))[0] for s in s_u]
    frames_h = [apply_heterogeneous_fog(img_bot, float(s))[0] for s in s_h]

    n = min(len(frames_u), len(frames_h))
    frames_u, frames_h = frames_u[:n], frames_h[:n]

    # Match widths for vertical stack
    target_w = min(frames_u[0].shape[1], frames_h[0].shape[1])
    stacked_frames = [
        cv2.vconcat([_resize_to_width(fu, target_w), _resize_to_width(fh, target_w)])
        for fu, fh in zip(frames_u, frames_h)
    ]

    H, W = stacked_frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(str(out_video), fourcc, fps, (W, H))
    if not vw.isOpened():
        raise RuntimeError("Could not open VideoWriter. Try out_video suffix '.avi' and FOURCC 'XVID'.")
    for f in stacked_frames:
        vw.write(f)
    vw.release()
    print(f"[OK] stacked video: {out_video} ({seconds}s @ {fps} FPS)")

    if also_save_individual:
        out_u = out_video.with_name(out_video.stem + "_uniform.mp4")
        out_h = out_video.with_name(out_video.stem + "_hetero.mp4")

        hu, wu = frames_u[0].shape[:2]
        hh, wh = frames_h[0].shape[:2]
        vw_u = cv2.VideoWriter(str(out_u), fourcc, fps, (wu, hu))
        vw_h = cv2.VideoWriter(str(out_h), fourcc, fps, (wh, hh))
        for fu in frames_u: vw_u.write(fu)
        for fh in frames_h: vw_h.write(fh)
        vw_u.release(); vw_h.release()
        print(f"[OK] individual videos: {out_u}, {out_h}")

# ----------- paths for your current structure -------------------------------
base = Path.cwd()                    # this notebook is inside examples/
images_dir = base / "images"
videos_dir = base / "videos"

top_img = images_dir / "example1.png"     # uniform on TOP
bot_img = images_dir / "example2.png"     # heterogeneous on BOTTOM
out_mp4 = videos_dir / "stack_uniform_vs_hetero.mp4"

make_stacked_uniform_vs_hetero_video(
    image_top=top_img,
    image_bottom=bot_img,
    out_video=out_mp4,
    seconds=5,
    fps=1,
    max_uniform=0.7,
    max_hetero=0.7,
    also_save_individual=False,
)


KeyboardInterrupt: 